# OpenSMOG Molecular Dynamics Simulation
###This notebook initializes and runs a structure-based molecular dynamics simulation using `OpenSMOG`.



## 1. Installing Packages
<br>
(A) SuBMIT: For generating coarse-grained SBM structure and forcefield files.<br>
(B) OpenMM: MD integrator.<br>
(C) OpenSMOG: OpenMM based SBM MD package.<br>
(D) MDTraj: For analyzing MD trajectory.<br>


In [ ]:
!git clone https://github.com/sglabncbs/SuBMIT
%cd /content/SuBMIT/
!pip install .


In [ ]:
!pip install openmm[cuda12] opensmog mdtraj py3Dmol


## 2. Generating C-alpha SBM files
###For C-alpha SBM simulations of 2CI2. <br><br>



###2.1 Run SuBMIT for 2CI2.pdb.


In [ ]:
# Go to examples directory
%cd /content/SuBMIT/examples/Clementi2000_CA-model/Default_LJ1012_contacts/OpenSMOG/
# Run submit-cli command with input pdb (--aa_pdb), SBM variant (--calpha_sbm) and tool type (--opensmog)
!submit-cli --aa_pdb 2CI2.pdb --calpha_sbm --opensmog


###2.2 View struture and plot native contact map.






In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol
import ipywidgets as widgets
from IPython.display import display

!ls 2CI2.pdb
# load and show 2CI2 structure
with open('2CI2.pdb','r') as fin:
    pdb_data = fin.read()
viewer = py3Dmol.view(width=500, height=500)
viewer.addModel(pdb_data, 'pdb')
viewer.setStyle({'cartoon': {'color': 'spectrum'}})
viewer.zoomTo()
struct_output = widgets.Output()
with struct_output:
  viewer.show()

# load and plot 2CI2 native contact map
!ls SuBMIT_Output/RenumberedPDB_CMap/prot.CGcont
with open('SuBMIT_Output/RenumberedPDB_CMap/prot.CGcont','r') as fin:
  C1,A1,C2,A2,W,D=np.transpose([l.split() for l in fin])
  A1,A2=np.intp([A1,A2])
  D=np.float64(D)
fig,ax=plt.subplots(1,1,figsize=(5.5, 5.5))
ax.set_aspect('1')
plot_output = widgets.Output()
plt.scatter(x=A1,y=A2,marker='s',s=10)
plt.scatter(x=A2,y=A1,marker='s',s=10)
plt.title("2CI2 Native Contact Map")
plt.xlabel("Residue number")
plt.ylabel("Residue number")

with plot_output:
  plt.show()

side_by_side_layout = widgets.HBox([struct_output,plot_output])
display(side_by_side_layout)



###2.3 Download SuBMIT_Output directory.




In [ ]:
!ls SuBMIT_Output/
!zip -r SuBMIT_Output.zip SuBMIT_Output/

from google.colab import files
files.download("SuBMIT_Output.zip")

## 3. Running SBM simulations

###3.1 Loading OpenSMOG python module

In [ ]:
from OpenSMOG import SBM


##3.2 Setting up simulation parameters

In [ ]:
simul_prefix = "Output"   # Output file prefix
dt = 0.0005               # time step in reduced time units (here picoseconds)
collision_rate = 1.0      # inverse time units (here ps^-1)
r_cutoff = 1.1            # nonbond interaction cutoff (in nm)
T = 0.98                  # Temperature in reduced units
# 2CI2 folding temperature is 0.98
report_interval=2000      # Write out after every 2000 steps
sbm_CA = SBM(name=simul_prefix,
             time_step=dt,
             collision_rate=collision_rate,
             r_cutoff=r_cutoff,
             temperature=T,
             pbc=True)    # Create simulation object


##3.3 Setting up simulation platform



In [ ]:

# Setup OpenMM platform (cuda, HIP, opencl, or cpu)
sbm_CA.setup_openmm(platform='cuda', GPUindex='default')



##3.4 Loading SuBMIT generated CG structure and forcefield files

In [ ]:
!ls SuBMIT_Output/opensmog.*
# file names
sbm_CA_grofile = "SuBMIT_Output/opensmog.gro"
sbm_CA_topfile = "SuBMIT_Output/opensmog.top"
sbm_CA_xmlfile = "SuBMIT_Output/opensmog.xml"
# Load files



##3.5 Create simulation context and setup energy, force and coordinates reporters

In [ ]:

# create simulation context
sbm_CA.loadSystem(Grofile=sbm_CA_grofile, Topfile=sbm_CA_topfile, Xmlfile=sbm_CA_xmlfile)
outdir = "T%.2f" % T
sbm_CA.saveFolder(outdir)
sbm_CA.createSimulation()
# Setup reporters
trjformat = "xtc"
sbm_CA.createReporters(trajectory=True, trajectoryFormat=trjformat,
                       energies=True, energy_components=True,
                       interval=report_interval)


##3.6 Run simulation


In [ ]:
# Run MD production steps
sbm_CA.run(nsteps=5000000, report=True, interval=10000)

# 4. Analysis

### Select trajectory

In [ ]:
trajectory_selected=False

(A) Pre-run.

In [ ]:

!git clone https://github.com/digvijaylp/SBMtutorial1.git
!ls ./SBMtutorial1/2CI2_CA-SBM/Simulations/T0.98/Output_trajectory.xtc
!ls ./SBMtutorial1/2CI2_CA-SBM/Simulations/T0.98/Output_energies.txt
sbm_CA_trjfile='./SBMtutorial1/2CI2_CA-SBM/Simulations/T0.98/Output_trajectory.xtc'
sbm_CA_edrfile='./SBMtutorial1/2CI2_CA-SBM/Simulations/T0.98/Output_energies.txt'
trajectory_selected=True

(B) Current-run.

In [ ]:
if trajectory_selected:
  print ("Already selected pre-run trajectory. Skipping !!")
else:
  sbm_CA_trjfile='T0.98/Output_trajectory.xtc'
  sbm_CA_edrfile='./T0.98/Output_energies.txt'

## 4.0 Loading the trajectory

In [ ]:
# check for trajectory
import mdtraj as md

# load trajectory
traj=md.load(sbm_CA_trjfile,top=sbm_CA_grofile)

print (traj)
sim_time=traj.time  # simulation time
sim_steps=sim_time/dt # simulation steps time/dt

with open(sbm_CA_edrfile) as fin:
  PE=np.float64([line.split(',')[1] for line in fin if not line.startswith('#')])
plt.plot(sim_steps,PE)
plt.xlabel("Simulation steps")
plt.ylabel("G(r)")
plt.title("Potential energy")

## 4.1 Radius of gyration

In [ ]:


# compute radius of gyration
Rg=md.compute_rg(traj=traj)

print ("Rg array shape:",Rg.shape)
print ("time-steps array shape:",sim_steps.shape)

plt.plot(sim_steps,Rg)
plt.xlabel("Simulation steps")
plt.ylabel("G(r)")
plt.title("Radius of Gyration")


## 4.2 Fraction native contacts

In [ ]:
# native contact map file
with open('SuBMIT_Output/RenumberedPDB_CMap/prot.CGcont','r') as fin:
  C1,A1,C2,A2,W,D=np.transpose([l.split() for l in fin])

# -1 to start atom index from 0
native_pairs=-1+np.intp([A1,A2]).T
# *0.1 to convert to nm
native_dists=0.1*np.float64(D)
print ("Shape of contact-map pairs array:",native_pairs.shape)
print ("Shape of contact-map distances array:",native_dists.shape)
total_contacts=native_dists.shape[0]


# calculate native pair distances in each frame
dists=md.compute_distances(traj=traj,periodic=True,atom_pairs=native_pairs)

# determing pairs which are 1.2*native distance in each frame
contacting_pairs=np.intp(dists<=1.2*native_dists) # 1 is contactsing, else 0

print ("Distance array shape (No.of.frames x No.of.pairs):",dists.shape)
print ("Contacting pairs array shape (No.of.frames x No.of.pairs):",dists.shape)

# total no. of contacting pairs (1s) per frame
Q=np.sum(contacting_pairs,1)
# convert to percent contacting native pairs (foldedness)
Q_percent=100*Q/total_contacts

print ("Q array shape (same as no.of.frames):",Q.shape)
plt.plot(sim_steps,Q_percent)
plt.xlabel("Simulation steps")
plt.ylabel("Folded-ness Q")
plt.title("Fraction native contacts")

## 4.3 Free energy as a function of Q

In [ ]:
print ("Q array shape (same as no.of.frames):",Q.shape)

n_bins=20
# 1-D array for Q-values (same as prev snippet)
bin_boundaries=np.linspace(0,total_contacts,n_bins)
#bin_width=10
#bin_boundaries=bin_width+np.intp(range(0,total_contacts,bin_width))
Q_to_bins=np.searchsorted(bin_boundaries,Q)
print (Q_to_bins)

# histogram
Q_hist=np.zeros(n_bins,dtype=int)
np.add.at(Q_hist,(Q_to_bins),1)

FEP=-np.log(Q_hist+1)
FEP=FEP-np.min(FEP)

X=100*bin_boundaries/total_contacts
fig, ax = plt.subplots()
plt.plot(X,FEP,linewidth=3,color="black")
plt.xlabel("Q",fontsize=22)
plt.ylabel("-ln(P(Q))",fontsize=22)
plt.xlim([0,100])
plt.ylim([0,6])
plt.xticks(fontsize = 22)
plt.yticks(fontsize = 22)
for axis in ['top','bottom','left','right']:
		ax.spines[axis].set_linewidth(3)
		ax.xaxis.set_tick_params(width=3, length=4)
		ax.yaxis.set_tick_params(width=3, length=4)



## 4.3(b) Reweight free energy profile

In [ ]:
def boltz_reWeight(Q,P,T_in_K,dt,data,tse):
	T1=T_in_K
	T2=T1
	threshold=0.004
	tse=int(np.sum(tse>Q))
	for x in range(0,1000):
		T2+=dt
		T2=np.round(T2,3)
		rew_data=data*np.exp(-P*(1.0/T2-1.0/T1)/R)
		s=np.sum(rew_data,1)
		lns=-np.log(s+1)
		lns-=min(lns)
		dlp=lns<=threshold
		uf_test=dlp[:tse]
		f_test=dlp[tse:]
		if sum(uf_test)>=1 and sum(f_test)>=1:
			return lns,T2
	return lns,T2

energy_bin_width=10
minp=int(np.floor(float(np.min(PE))/energy_bin_width)*energy_bin_width)
maxp=int(np.ceil(float(np.max(PE))/energy_bin_width)*energy_bin_width)
PE_bin_boundaries=energy_bin_width+np.intp(range(minp,maxp,energy_bin_width))
n_PEbins=PE_bin_boundaries.shape[0]
PE_to_bins=np.searchsorted(PE_bin_boundaries,PE)
Q_PE_hist=np.zeros((n_bins,n_PEbins),dtype=int)
np.add.at(Q_PE_hist,(Q_to_bins,PE_to_bins),1)

R= 0.008314  # KJ mol-1 K-1
FEP,TF=boltz_reWeight(data=Q_PE_hist,
                      Q=bin_boundaries,
                      P=PE_bin_boundaries,
                      T_in_K=T/R,
                      dt=-0.001,
                      tse=total_contacts/2)
TF*=R
print (T,TF)


X=100*bin_boundaries/total_contacts
fig, ax = plt.subplots()
plt.plot(X,FEP,linewidth=3,color="black")
plt.xlabel("Q",fontsize=22)
plt.ylabel("-ln(P(Q))",fontsize=22)
plt.xlim([0,100])
plt.ylim([0,6])
plt.xticks(fontsize = 22)
plt.yticks(fontsize = 22)
for axis in ['top','bottom','left','right']:
		ax.spines[axis].set_linewidth(3)
		ax.xaxis.set_tick_params(width=3, length=4)
		ax.yaxis.set_tick_params(width=3, length=4)





## 4.4 Average contact maps (Predicting folding pathway(s))

In [ ]:
!pip install imageio[ffmpeg]
import os
import imageio.v3 as iio
from IPython.display import Video, display

# Create a temporary folder to hold the frames
os.makedirs('ACM',exist_ok=True)


In [ ]:
print ("Contacting pairs array shape (No.of.frames x No.of.pairs):",dists.shape)
print ("Q array shape (same as no.of.frames):",Q.shape)

print (n_bins)
# 2-D array for Q-values and corresponding contacting pairs
pairs_hist=np.zeros((n_bins,total_contacts),dtype=int)
print ("ACM histogram array shape",pairs_hist.shape)

def acm_plot(X,Y,Z,FEP,bi):
  fig,(ax1,ax2)=plt.subplots(1, 2, figsize=(16, 7))
  #fig,ax=plt.subplots(1,1)
  # FEP for reference
  X1=100*bin_boundaries/total_contacts

  ax1.plot(X1,FEP,linewidth=3, color="black")
  ax1.axvspan(X1[bi-1],X1[bi],color='yellow',alpha=0.3,zorder=0)

  ax1.set_xlabel("Q",fontsize=22)
  ax1.set_ylabel("-ln(P(Q))",fontsize=22)
  ax1.set_xlim([0, 100])
  ax1.set_ylim([0, 6])
  ax1.tick_params(axis='both',which='major',labelsize=22)
  for axis in ['top','bottom','left','right']:
      ax1.spines[axis].set_linewidth(3)
  ax1.xaxis.set_tick_params(width=3,length=4)
  ax1.yaxis.set_tick_params(width=3,length=4)

  # ACM
  ax2.set_aspect('1')
  sc1=ax2.scatter(x=X,y=Y,c=Z,marker="s",s=25,cmap='hot_r')
  sc2=ax2.scatter(x=Y,y=X,c=Z,marker="s",s=25,cmap='hot_r')
  ax2.set_xlim([1,max(X)])
  ax2.set_ylim([1,max(Y)])
  sc1.set_clim([0,1])
  #cbar=plt.colorbar(aspect='8')
  cbar=fig.colorbar(sc1,ax=ax2,aspect=8)
  cbar.ax.tick_params(width=4,labelsize=22,length=8)
  cbar.set_ticks([0,1])
  cbar.outline.set_linewidth(4)
  ax2.tick_params(width=4,labelsize=22,length=8)

  ax2.set_xlim([0,65])
  ax2.set_ylim([0,65])
  ax2.set_xticks(range(0,70,10))
  ax2.set_yticks(range(0,70,10))
  for i in ['top','bottom','left','right']:
      ax2.spines[i].set_linewidth(5)
  ax2.set_xlabel("Residue number",fontsize=22)
  ax2.set_ylabel("Residue number",fontsize=22)

  filename = f"ACM/acm_{bi:03d}.png"
  plt.savefig(filename, bbox_inches='tight', dpi=100)
  return filename




# Contacting pairs frame and pair indices
frame_idx,pair_idx=np.where(dists<=1.2*native_dists)
# get bin index for the frames
bin_idx=Q_to_bins[frame_idx]
print ("Bin index array shape:",bin_idx.shape)
print ("Pair index array shape:",pair_idx.shape)

# add in histogram
np.add.at(pairs_hist,(bin_idx,pair_idx),1)

# plot
plt.ioff()
frame_files = []
X,Y=native_pairs.T + 1 # adding 1
for j in range(1,n_bins):
  if Q_hist[j]==0: continue
  Z=pairs_hist[j]/Q_hist[j]
  filename=acm_plot(X,Y,Z,FEP,j)
  frame_files.append(filename)

print("All frames generated. Compiling video...")
print (frame_files)
images = [iio.imread(f) for f in frame_files]
iio.imwrite('acm_plots_video.mp4', images, fps=2)
display(Video('acm_plots_video.mp4', embed=True, width=800))
